Link: [Ultimate RAG Bootcamp Using Langchain,LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

# Query Enhancement – Query Decomposition

## 📚 Learning Objectives
By the end of this notebook, you will understand:
1. What query decomposition is and when to use it
2. How to break complex questions into simpler sub-questions
3. How to build a RAG pipeline that answers each sub-question

---

## 🧠 What is Query Decomposition?

Query decomposition is the process of taking a **complex, multi-part question** and breaking it into **simpler, atomic sub-questions** that can each be retrieved and answered individually.

### 🔄 How It Works

```
Complex Query: "How does LangChain use memory and agents compared to CrewAI?"
                                    ↓
                          [LLM Decomposition]
                                    ↓
        ┌───────────────────────────┼───────────────────────────┐
        ↓                           ↓                           ↓
"How does LangChain      "How does LangChain      "How does CrewAI
 use memory?"             use agents?"             handle memory
                                                   and agents?"
        ↓                           ↓                           ↓
  [Retrieve + Answer]        [Retrieve + Answer]        [Retrieve + Answer]
        ↓                           ↓                           ↓
        └───────────────────────────┼───────────────────────────┘
                                    ↓
                           [Combine Answers]
```

### ✅ Why Use Query Decomposition?

| Benefit | Explanation |
|---------|-------------|
| **Better Retrieval** | Simple questions are easier to match against documents |
| **Multi-hop Reasoning** | Complex questions requiring multiple pieces of information |
| **Completeness** | Ensures all parts of the question are addressed |
| **Parallelization** | Sub-questions can be processed in parallel |
| **Debugging** | Easier to identify which part of the answer is wrong |

### 📊 When to Use Query Decomposition

✅ **Good for:**
- Questions with multiple parts ("What is X and how does it compare to Y?")
- Questions requiring multi-hop reasoning
- Questions about relationships between concepts

❌ **Not needed for:**
- Simple, single-concept questions
- Yes/no questions
- Questions with clear, focused intent


# Query Enhancement – Query Decomposition

## 📚 Learning Objectives
By the end of this notebook, you will understand:
1. What query decomposition is and when to use it
2. How to break complex questions into simpler sub-questions
3. How to build a RAG pipeline that answers each sub-question

---

## 🧠 What is Query Decomposition?

Query decomposition is the process of taking a **complex, multi-part question** and breaking it into **simpler, atomic sub-questions** that can each be retrieved and answered individually.

### 🔄 How It Works

```
Complex Query: "How does LangChain use memory and agents compared to CrewAI?"
                                    ↓
                          [LLM Decomposition]
                                    ↓
        ┌───────────────────────────┼───────────────────────────┐
        ↓                           ↓                           ↓
"How does LangChain      "How does LangChain      "How does CrewAI
 use memory?"             use agents?"             handle memory
                                                   and agents?"
        ↓                           ↓                           ↓
  [Retrieve + Answer]        [Retrieve + Answer]        [Retrieve + Answer]
        ↓                           ↓                           ↓
        └───────────────────────────┼───────────────────────────┘
                                    ↓
                           [Combine Answers]
```

### ✅ Why Use Query Decomposition?

| Benefit | Explanation |
|---------|-------------|
| **Better Retrieval** | Simple questions are easier to match against documents |
| **Multi-hop Reasoning** | Complex questions requiring multiple pieces of information |
| **Completeness** | Ensures all parts of the question are addressed |
| **Parallelization** | Sub-questions can be processed in parallel |
| **Debugging** | Easier to identify which part of the answer is wrong |

### 📊 When to Use Query Decomposition

✅ **Good for:**
- Questions with multiple parts ("What is X and how does it compare to Y?")
- Questions requiring multi-hop reasoning
- Questions about relationships between concepts

❌ **Not needed for:**
- Simple, single-concept questions
- Yes/no questions
- Questions with clear, focused intent

In [ ]:
# ============================================================================
# STEP 0: Import Required Libraries
# ============================================================================
# 
# Components needed for query decomposition RAG:
# - LLM for decomposing queries and generating answers
# - Document loading and splitting for knowledge base
# - Embeddings and vector store for semantic search
# - Chain utilities for building the pipeline
# ============================================================================

from langchain.chat_models import init_chat_model          # Initialize LLM
from langchain.prompts import PromptTemplate               # Create prompts
from langchain.document_loaders import TextLoader          # Load text files
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Split text
from langchain_huggingface import HuggingFaceEmbeddings    # Local embeddings
from langchain_community.vectorstores import FAISS         # Vector store
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain.chains.combine_documents import create_stuff_documents_chain  # Combine docs
from langchain_core.runnables import RunnableSequence      # Chain operations

In [ ]:
# ============================================================================
# STEP 1: Load, Embed, and Create Retriever
# ============================================================================
# 
# We create our knowledge base by:
# 1. Loading the text file containing LangChain and CrewAI information
# 2. Splitting into smaller chunks for better retrieval granularity
# 3. Creating embeddings using a local HuggingFace model
# 4. Building a FAISS vector store for fast similarity search
# 5. Creating an MMR retriever for diverse, relevant results
#
# MMR Parameters:
# - k=4: Return 4 documents per query
# - lambda_mult=0.7: 70% relevance, 30% diversity balance
#   (Higher = more relevant but potentially similar docs)
# ============================================================================

# Load the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
print(f"📄 Loaded {len(docs)} document(s)")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"📦 Created {len(chunks)} chunks")

# Create embeddings and vector store
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
print("✅ Vector store created")

# Create MMR retriever for diverse results
# lambda_mult controls relevance vs diversity trade-off
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,              # Return 4 documents
        "lambda_mult": 0.7   # 70% relevance, 30% diversity
    }
)
print("✅ MMR retriever ready")

In [ ]:
# ============================================================================
# STEP 2: Initialize the LLM
# ============================================================================
# 
# We use Groq's Llama 3.1 8B model, which offers:
# - Very fast inference (Groq's LPU architecture)
# - Free tier available for development
# - Good quality for decomposition and answering tasks
#
# 💡 The LLM will be used for:
#    1. Decomposing complex queries into sub-questions
#    2. Answering each sub-question based on retrieved context
# ============================================================================

import os
from dotenv import load_dotenv
load_dotenv()

# Set the API key from environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Groq's fast Llama model
llm = init_chat_model(model="groq:llama-3.1-8b-instant")
print("✅ LLM initialized (Groq Llama 3.1 8B)")
llm

In [ ]:
# ============================================================================
# STEP 3: Create the Query Decomposition Chain
# ============================================================================
# 
# This is the CORE component of query decomposition!
# 
# The prompt instructs the LLM to:
# - Analyze the complex question
# - Break it into 2-4 simpler, focused sub-questions
# - Each sub-question should be self-contained and answerable
#
# 🎯 Key Design Decisions:
# - Limit to 2-4 sub-questions to avoid over-decomposition
# - Each sub-question should focus on ONE concept
# - Sub-questions should collectively cover the original query
#
# The chain: prompt → LLM → string output parser
# ============================================================================

decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

# Create the decomposition chain using LCEL
decomposition_chain = decomposition_prompt | llm | StrOutputParser()
print("✅ Query decomposition chain created")

In [ ]:
# ============================================================================
# 🧪 TEST: See Query Decomposition in Action
# ============================================================================
# 
# Let's test the decomposition chain with a complex, multi-part question.
# This question involves:
# - LangChain memory
# - LangChain agents
# - CrewAI comparison
# 
# The LLM should break this into simpler sub-questions.
# ============================================================================

query = "How does LangChain use memory and agents compared to CrewAI?"
print(f"📝 Original Query:\n{query}\n")
print("🔄 Decomposing query...\n")

decomposition_question = decomposition_chain.invoke({"question": query})


In [ ]:
# Display the decomposed sub-questions
# The LLM breaks down the complex query into focused, answerable parts
print("📋 Decomposed Sub-questions:")
print("="*60)
print(decomposition_question)

In [ ]:
# ============================================================================
# STEP 4: Create the Question-Answering Chain
# ============================================================================
# 
# This chain answers EACH sub-question individually using retrieved context.
# 
# For each sub-question:
# 1. Retrieve relevant documents from the vector store
# 2. "Stuff" the documents into the prompt as context
# 3. Generate an answer based on the context
#
# This approach ensures each sub-question gets focused, relevant context
# rather than trying to answer everything at once.
# ============================================================================

qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

# Create the QA chain that will answer each sub-question
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)
print("✅ Question-answering chain created")

In [ ]:
# ============================================================================
# STEP 5: Build the Complete Query Decomposition RAG Pipeline
# ============================================================================
# 
# This function orchestrates the entire process:
#
# 1. DECOMPOSE: Break the complex query into sub-questions
# 2. For EACH sub-question:
#    a. RETRIEVE: Get relevant documents from vector store
#    b. ANSWER: Generate answer using retrieved context
# 3. COMBINE: Collect all Q&A pairs into final response
#
# Architecture:
# ┌─────────────────────────────────────────────────────────────┐
# │                Query Decomposition Pipeline                  │
# │                                                              │
# │  Complex Query → [Decompose] → Sub-Q1, Sub-Q2, Sub-Q3...    │
# │                                    │                         │
# │                     ┌──────────────┼──────────────┐          │
# │                     ↓              ↓              ↓          │
# │              [Retrieve]      [Retrieve]      [Retrieve]      │
# │                     ↓              ↓              ↓          │
# │               [Answer]       [Answer]        [Answer]        │
# │                     ↓              ↓              ↓          │
# │                     └──────────────┼──────────────┘          │
# │                                    ↓                         │
# │                          [Combine Results]                   │
# └─────────────────────────────────────────────────────────────┘
# ============================================================================

def full_query_decomposition_rag_pipeline(user_query):
    """
    Execute query decomposition RAG pipeline.
    
    Args:
        user_query: The complex question to answer
        
    Returns:
        Combined answers for all sub-questions
    """
    print(f"📝 Original Query: {user_query}\n")
    
    # Step 1: Decompose the query into sub-questions
    print("🔄 Step 1: Decomposing query...")
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    
    # Parse sub-questions (remove bullets, numbers, etc.)
    sub_questions = [
        q.strip("-•1234567890. ").strip() 
        for q in sub_qs_text.split("\n") 
        if q.strip()
    ]
    print(f"   Found {len(sub_questions)} sub-questions\n")
    
    # Step 2: Answer each sub-question
    print("🔍 Step 2: Answering each sub-question...")
    results = []
    for i, subq in enumerate(sub_questions, 1):
        print(f"   Processing sub-question {i}/{len(sub_questions)}...")
        
        # Retrieve relevant documents for this sub-question
        docs = retriever.invoke(subq)
        
        # Generate answer using retrieved context
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    # Combine all answers
    return "\n\n".join(results)

print("✅ Query decomposition RAG pipeline ready!")

In [ ]:
# ============================================================================
# STEP 6: Run the Complete Pipeline
# ============================================================================
# 
# Let's test our query decomposition pipeline with the complex question.
# Watch how it:
# 1. Breaks down the question into parts
# 2. Answers each part individually
# 3. Combines everything for a comprehensive response
# ============================================================================

query = "How does LangChain use memory and agents compared to CrewAI?"

print("="*70)
print("🚀 RUNNING QUERY DECOMPOSITION RAG PIPELINE")
print("="*70 + "\n")

final_answer = full_query_decomposition_rag_pipeline(query)

print("\n" + "="*70)
print("✅ FINAL COMPREHENSIVE ANSWER")
print("="*70 + "\n")
print(final_answer)

## 📊 Summary: Query Decomposition Benefits

### Comparison: With vs Without Decomposition

| Aspect | Standard RAG | Decomposition RAG |
|--------|-------------|-------------------|
| **Complex Query Handling** | May miss parts of multi-concept questions | Each concept addressed individually |
| **Retrieval Precision** | Single retrieval for complex query | Targeted retrieval per sub-question |
| **Answer Completeness** | Often incomplete | Comprehensive coverage |
| **Debugging** | Hard to identify failures | Easy to see which sub-answer failed |

## 🎯 Key Takeaways

1. **Decomposition uses an LLM** to break complex questions into simpler parts
2. **Each sub-question gets its own retrieval** and answer generation
3. **Results are combined** for a comprehensive final answer
4. **Best for**: Multi-part questions, comparisons, multi-hop reasoning

## ⚠️ Considerations

- **Latency**: Multiple retrieval + LLM calls increase total time
- **Cost**: More API calls mean higher costs
- **Over-decomposition**: Too many sub-questions can fragment context
- **Parsing Challenges**: LLM output format may vary

## 🔧 Optimization Tips

1. **Parallel Processing**: Run sub-question RAG calls in parallel
2. **Caching**: Cache decomposition results for repeated queries
3. **Limit Sub-questions**: Keep to 2-4 for best balance
4. **Final Synthesis**: Add an optional step to synthesize sub-answers into coherent response

## 🔗 Related Techniques

- **Query Expansion**: Enrich queries with synonyms (previous notebook)
- **HyDE**: Generate hypothetical answers for better retrieval (next notebook)
- **Step-back Prompting**: Ask a more general question first
